# Cálculo 3 — Visualizações Interativas

Tópicos de **Integrais Múltiplas, de Linha e de Superfície**, construídos passo a passo a partir da ideia geométrica até a fórmula final:

**Parte 1 — Integrais Duplas**
1. **Volume por Integral Dupla** — somas de Riemann (prismas retangulares) convergindo para $\iint_D f(x,y)\,dA$
2. **Teorema de Fubini** — troca da ordem de integração, $\displaystyle\int\int f\,dy\,dx = \int\int f\,dx\,dy$
3. **Massa e Momento de Inércia** — $M$, $I_0$, $I_x$, $I_y$ de uma lâmina com densidade $\rho(x,y)$

**Parte 2 — Integrais de Linha**
4. **Integrais de Linha e Comprimento de Curvas** — $L=\int_a^b|\mathbf r'(t)|\,dt$ e $\int_C f\,ds$
5. **Teorema de Green** — $\displaystyle\oint_C P\,dx+Q\,dy = \iint_D\left(\frac{\partial Q}{\partial x}-\frac{\partial P}{\partial y}\right)dA$
6. **Trabalho de Campos de Força** — campos conservativos vs. não conservativos e independência de caminho

**Parte 3 — Integrais de Superfície**
7. **Representação Paramétrica de Superfícies** — plano, esfera, cone e toro via $\mathbf r(u,v)$
8. **Integrais de Superfície** — área $A=\iint_D|\mathbf r_u\times\mathbf r_v|\,dA$ e $\iint_S f\,dS$

> **Como usar:** cada seção tem sua própria célula de parâmetros — edite as strings e constantes marcadas com `# === Parametros ===` e reexecute.

---

## Como usar este notebook

1. **Cada seção é independente:** diferente do Cálculo 2 (uma única função global), aqui cada tópico tem suas próprias variáveis configuráveis (função, região, campo vetorial ou parametrização de superfície).
2. **Execute sempre a Seção 0 primeiro:** ela define os imports, os símbolos simbólicos (`x, y, z, t, u, v`) e as funções utilitárias reaproveitadas pelas demais seções.
3. **Sintaxe (SymPy):** use `**` para potência, `sin`, `cos`, `exp`, `log`, `sqrt`, `Abs`, `pi`.
4. **Plotly é opcional:** se não estiver instalado (`pip install plotly`), os gráficos interativos 3-D são pulados automaticamente e apenas os gráficos estáticos (Matplotlib) são exibidos.

## 0. Configuração

Imports, símbolos globais e funções utilitárias usados em todas as seções.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sympy as sp
from matplotlib import cm
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display

try:
    import plotly.graph_objects as go

    PLOTLY = True
except ImportError:
    PLOTLY = False
    print("plotly nao instalado — execute: pip install plotly")

matplotlib.style.use("ggplot")
%matplotlib inline

# Simbolos globais usados em todas as secoes
x, y, z, t, u, v = sp.symbols("x y z t u v", real=True)
_SIMBOLOS = {"x": x, "y": y, "z": z, "t": t, "u": u, "v": v}


def parse(expr_str):
    '''Converte uma string em expressao SymPy reutilizando os simbolos globais
    acima, para que sp.diff/sp.integrate reconhecam corretamente as variaveis.'''
    return sp.sympify(expr_str, locals=_SIMBOLOS)


def avalia(expr_sym, variaveis, arrays):
    '''Avalia uma expressao simbolica sobre um ou mais arrays numpy, cuidando do
    caso em que o resultado nao depende de alguma variavel (broadcast automatico
    para o formato determinado pelos arrays de entrada).'''
    fn = sp.lambdify(variaveis, expr_sym, "numpy")
    val = np.asarray(fn(*arrays), dtype=float)
    alvo = np.broadcast_shapes(*(np.shape(a) for a in arrays))
    if val.shape != alvo:
        val = np.broadcast_to(val, alvo).copy()
    return val


def integral_numerica(expr_sym, var, lo, hi):
    '''Integra uma expressao simbolica em [lo, hi]. Se o SymPy nao encontrar uma
    antiderivada fechada, cai automaticamente para quadratura numerica (mpmath)
    ao chamar .evalf() sobre o objeto Integral.'''
    return float(sp.Integral(expr_sym, (var, lo, hi)).evalf())


def integral_dupla(expr_sym, var_int, lo_int, hi_int, var_ext, lo_ext, hi_ext):
    '''Integral dupla iterada: integra primeiro em var_int (tentando uma forma
    fechada com sp.integrate) e depois em var_ext (numericamente, via
    integral_numerica). Mais robusto do que pedir ao SymPy uma unica integral
    com dois pares de limites, que pode falhar numericamente.'''
    interna = sp.integrate(expr_sym, (var_int, lo_int, hi_int))
    return integral_numerica(sp.simplify(interna), var_ext, lo_ext, hi_ext)


def paredes_verticais(ax, xs, ys, zs, cor="steelblue", alpha=0.3):
    '''Desenha "paredes" verticais (de z=0 ate zs) ao longo do caminho (xs, ys),
    usadas para visualizar o solido preenchido sob uma superficie.'''
    verts = []
    for i in range(len(xs) - 1):
        verts.append(
            [
                (xs[i], ys[i], 0.0),
                (xs[i + 1], ys[i + 1], 0.0),
                (xs[i + 1], ys[i + 1], zs[i + 1]),
                (xs[i], ys[i], zs[i]),
            ]
        )
    ax.add_collection3d(Poly3DCollection(verts, color=cor, alpha=alpha, edgecolor="none"))


print("Ambiente de Calculo 3 configurado.")
print(f"Plotly disponivel para graficos 3-D interativos: {PLOTLY}")

## 1. Volume por Integral Dupla — Somas de Riemann

Para $f(x,y)\ge 0$ numa região retangular $D=[a,b]\times[c,d]$, o **volume** do sólido entre o gráfico de $f$ e o plano $z=0$ é aproximado dividindo $D$ em pequenos retângulos $\Delta A_{ij}=\Delta x\,\Delta y$ e somando o volume de prismas de altura $f(x_{ij}^*, y_{ij}^*)$ (ponto médio de cada célula):

$$V \approx \sum_{i=1}^{m}\sum_{j=1}^{n} f(x_{ij}^*, y_{ij}^*)\,\Delta A_{ij}$$

Quando $m,n\to\infty$ (e $\Delta x,\Delta y\to 0$), a soma de Riemann converge para a **integral dupla**:

$$V=\iint_D f(x,y)\,dA=\int_a^b\int_c^d f(x,y)\,dy\,dx$$

Abaixo, a soma de Riemann é calculada para $n$ crescente (mostrando a convergência) e depois o sólido é desenhado por completo (superfície + paredes laterais).

In [ ]:
# === Parametros ===
expr_str_vol = "4 - x**2 - y**2"  # f(x,y) >= 0 na regiao de integracao (altura do solido)
a_vol, b_vol = -1.0, 1.0  # limites de integracao em x
c_vol, d_vol = -1.0, 1.0  # limites de integracao em y (regiao retangular [a,b]x[c,d])
n_tiras = [2, 4, 10, 30]  # numero de subdivisoes por eixo, mostrando a convergencia
# ==================

f_vol_sym = parse(expr_str_vol)
f_vol_num = sp.lambdify((x, y), f_vol_sym, "numpy")

print("Volume do solido sob f(x,y) =")
display(f_vol_sym)

vol_exato = sp.integrate(sp.integrate(f_vol_sym, (y, c_vol, d_vol)), (x, a_vol, b_vol))
vol_exato_f = float(vol_exato)
print(f"\nVolume exato (integral dupla): V = {sp.nsimplify(vol_exato)} = {vol_exato_f:.5f}")

# --- Soma de Riemann (ponto medio de cada prisma), para n crescente ---
fig, axes = plt.subplots(
    1, len(n_tiras), figsize=(4.3 * len(n_tiras), 4.3), dpi=100,
    facecolor="w", subplot_kw={"projection": "3d"},
)
riemann_vals = []
for ax, n in zip(axes, n_tiras):
    xs = np.linspace(a_vol, b_vol, n + 1)
    ys = np.linspace(c_vol, d_vol, n + 1)
    dx = (b_vol - a_vol) / n
    dy = (d_vol - c_vol) / n
    soma = 0.0
    for i in range(n):
        xm = 0.5 * (xs[i] + xs[i + 1])
        for j in range(n):
            ym = 0.5 * (ys[j] + ys[j + 1])
            h = float(f_vol_num(xm, ym))
            soma += h * dx * dy
            ax.bar3d(
                xs[i], ys[j], 0, dx, dy, h,
                color="steelblue", alpha=0.65, edgecolor="k", linewidth=0.15, shade=True,
            )
    riemann_vals.append(soma)
    ax.set_title(f"n = {n}x{n}\nsoma = {soma:.4f}", fontsize=10, fontweight="bold")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

plt.suptitle(f"Convergencia da soma de Riemann para V = {vol_exato_f:.4f}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nConvergencia da soma de Riemann:")
for n, s_val in zip(n_tiras, riemann_vals):
    print(f"  n={n:3d}x{n:<3d}: soma = {s_val:.6f}   erro = {abs(s_val - vol_exato_f):.6f}")

# --- Solido preenchido (superficie + base + paredes laterais) ---
Xs, Ys = np.meshgrid(np.linspace(a_vol, b_vol, 60), np.linspace(c_vol, d_vol, 60))
Zs = avalia(f_vol_sym, (x, y), (Xs, Ys))

fig = plt.figure(figsize=(9, 7), dpi=100, facecolor="w")
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xs, Ys, Zs, cmap="viridis", alpha=0.92, rstride=2, cstride=2, linewidth=0)

borda_x = np.linspace(a_vol, b_vol, 60)
borda_y = np.linspace(c_vol, d_vol, 60)
paredes_verticais(ax, borda_x, np.full_like(borda_x, c_vol),
                   avalia(f_vol_sym, (x, y), (borda_x, np.full_like(borda_x, c_vol))))
paredes_verticais(ax, borda_x, np.full_like(borda_x, d_vol),
                   avalia(f_vol_sym, (x, y), (borda_x, np.full_like(borda_x, d_vol))))
paredes_verticais(ax, np.full_like(borda_y, a_vol), borda_y,
                   avalia(f_vol_sym, (x, y), (np.full_like(borda_y, a_vol), borda_y)))
paredes_verticais(ax, np.full_like(borda_y, b_vol), borda_y,
                   avalia(f_vol_sym, (x, y), (np.full_like(borda_y, b_vol), borda_y)))

ax.set_title(f"Solido sob f(x,y) = {expr_str_vol}", fontsize=12, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.tight_layout()
plt.show()

if PLOTLY:
    fig_pl = go.Figure(
        data=[
            go.Surface(x=Xs, y=Ys, z=Zs, colorscale="Viridis", opacity=0.95, showscale=False),
            go.Surface(
                x=Xs, y=Ys, z=np.zeros_like(Zs),
                colorscale=[[0, "lightsteelblue"], [1, "lightsteelblue"]],
                opacity=0.35, showscale=False,
            ),
        ]
    )
    fig_pl.update_layout(
        title=f"Solido interativo sob f(x,y) = {expr_str_vol} (arraste para girar)",
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        margin=dict(l=0, r=0, b=0, t=45),
        height=550,
    )
    fig_pl.show()
else:
    print("Instale plotly para visualizacao 3-D interativa: pip install plotly")

## 2. Teorema de Fubini — Troca da Ordem de Integração

Se $f$ é contínua numa região $D$ que pode ser descrita tanto como **tipo I** ($a\le x\le b,\ g_1(x)\le y\le g_2(x)$) quanto como **tipo II** ($c\le y\le d,\ h_1(y)\le x\le h_2(y)$), o Teorema de Fubini garante:

$$\iint_D f\,dA=\int_a^b\!\!\int_{g_1(x)}^{g_2(x)} f(x,y)\,dy\,dx=\int_c^d\!\!\int_{h_1(y)}^{h_2(y)} f(x,y)\,dx\,dy$$

Abaixo, a mesma região $D$ é descrita das duas formas e a integral de $f$ é calculada nos dois sentidos — os resultados devem coincidir. A região é visualizada com "fatias" verticais (ordem $dy\,dx$) e horizontais (ordem $dx\,dy$).

In [ ]:
# === Parametros ===
expr_str_fub = "x*y + 1"  # f(x,y)
# Regiao D, tipo I:  a <= x <= b,  y_lo(x) <= y <= y_hi(x)
a_f, b_f = 0.0, 1.0
y_lo_str, y_hi_str = "0", "x"
# Mesma regiao D, tipo II (equivalente):  c <= y <= d,  x_lo(y) <= x <= x_hi(y)
c_f, d_f = 0.0, 1.0
x_lo_str, x_hi_str = "y", "1"
# ==================

f_fub_sym = parse(expr_str_fub)
y_lo_sym, y_hi_sym = parse(y_lo_str), parse(y_hi_str)
x_lo_sym, x_hi_sym = parse(x_lo_str), parse(x_hi_str)

print("Integrando f(x,y) =")
display(f_fub_sym)

print("\nOrdem dy dx  (regiao tipo I):")
interno_1 = sp.integrate(f_fub_sym, (y, y_lo_sym, y_hi_sym))
display(sp.Eq(sp.Integral(f_fub_sym, (y, y_lo_sym, y_hi_sym)), interno_1))
resultado_1 = sp.integrate(interno_1, (x, a_f, b_f))
display(sp.Eq(sp.Integral(interno_1, (x, a_f, b_f)), resultado_1))

print("\nOrdem dx dy  (regiao tipo II):")
interno_2 = sp.integrate(f_fub_sym, (x, x_lo_sym, x_hi_sym))
display(sp.Eq(sp.Integral(f_fub_sym, (x, x_lo_sym, x_hi_sym)), interno_2))
resultado_2 = sp.integrate(interno_2, (y, c_f, d_f))
display(sp.Eq(sp.Integral(interno_2, (y, c_f, d_f)), resultado_2))

diferenca = sp.simplify(resultado_1 - resultado_2)
print(f"\ndy dx = {resultado_1}   dx dy = {resultado_2}")
if diferenca == 0:
    print("Fubini confirmado: os dois resultados coincidem!")
else:
    print(f"Atencao: os resultados diferem por {diferenca} — revise os limites da regiao.")

# --- Visualizacao da regiao D nas duas formas (tiras verticais e horizontais) ---
Xr = np.linspace(a_f, b_f, 300)
Ylo_x = avalia(y_lo_sym, (x,), (Xr,))
Yhi_x = avalia(y_hi_sym, (x,), (Xr,))

Yr = np.linspace(c_f, d_f, 300)
Xlo_y = avalia(x_lo_sym, (y,), (Yr,))
Xhi_y = avalia(x_hi_sym, (y,), (Yr,))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=100, facecolor="w")

ax = axes[0]
ax.fill_between(Xr, Ylo_x, Yhi_x, color="steelblue", alpha=0.3)
for xi in np.linspace(a_f, b_f, 9)[1:-1]:
    ylo_i = float(avalia(y_lo_sym, (x,), (np.array([xi]),))[0])
    yhi_i = float(avalia(y_hi_sym, (x,), (np.array([xi]),))[0])
    ax.plot([xi, xi], [ylo_i, yhi_i], color="darkblue", linewidth=1.6)
    ax.plot(xi, yhi_i, marker="^", color="darkblue", markersize=5)
ax.set_title("Tipo I: fatias verticais  (ordem dy dx)", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")

ax = axes[1]
ax.fill_betweenx(Yr, Xlo_y, Xhi_y, color="indianred", alpha=0.3)
for yi in np.linspace(c_f, d_f, 9)[1:-1]:
    xlo_i = float(avalia(x_lo_sym, (y,), (np.array([yi]),))[0])
    xhi_i = float(avalia(x_hi_sym, (y,), (np.array([yi]),))[0])
    ax.plot([xlo_i, xhi_i], [yi, yi], color="darkred", linewidth=1.6)
    ax.plot(xhi_i, yi, marker=">", color="darkred", markersize=5)
ax.set_title("Tipo II: fatias horizontais  (ordem dx dy)", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")

plt.suptitle(f"Regiao D para f(x,y) = {expr_str_fub}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Massa e Momento de Inércia de uma Lâmina

Uma lâmina plana ocupando a região $D$ com densidade superficial $\rho(x,y)$ (massa por área) tem:

$$\text{Massa: } M=\iint_D \rho(x,y)\,dA$$

$$\text{Momentos de inércia: } I_x=\iint_D y^2\rho\,dA,\qquad I_y=\iint_D x^2\rho\,dA,\qquad I_0=I_x+I_y=\iint_D (x^2+y^2)\rho\,dA$$

$I_0$ (momento polar) mede a resistência da lâmina à rotação em torno da origem (eixo $z$); $I_x$ e $I_y$ medem a resistência à rotação em torno dos eixos $x$ e $y$, respectivamente.

In [ ]:
# === Parametros ===
rho_str = "1 + x**2 + y**2"  # densidade rho(x,y)
a_m, b_m = 0.0, 1.0
y_lo_m_str, y_hi_m_str = "0", "x"  # regiao D, tipo I (mesmo formato da Secao 2)
# ==================

rho_sym = parse(rho_str)
y_lo_m, y_hi_m = parse(y_lo_m_str), parse(y_hi_m_str)

print("Densidade rho(x,y) =")
display(rho_sym)

M_sym = sp.integrate(sp.integrate(rho_sym, (y, y_lo_m, y_hi_m)), (x, a_m, b_m))
Ix_sym = sp.integrate(sp.integrate(y**2 * rho_sym, (y, y_lo_m, y_hi_m)), (x, a_m, b_m))
Iy_sym = sp.integrate(sp.integrate(x**2 * rho_sym, (y, y_lo_m, y_hi_m)), (x, a_m, b_m))
I0_sym = sp.simplify(Ix_sym + Iy_sym)

print("\nMassa:")
display(sp.Eq(sp.Symbol("M"), M_sym))
print("Momento de inercia em relacao ao eixo x:")
display(sp.Eq(sp.Symbol("I_x"), Ix_sym))
print("Momento de inercia em relacao ao eixo y:")
display(sp.Eq(sp.Symbol("I_y"), Iy_sym))
print("Momento polar (em relacao a origem):")
display(sp.Eq(sp.Symbol("I_0"), I0_sym))

M_f, Ix_f, Iy_f, I0_f = float(M_sym), float(Ix_sym), float(Iy_sym), float(I0_sym)
print(f"\nM = {M_f:.5f}   I_x = {Ix_f:.5f}   I_y = {Iy_f:.5f}   I_0 = {I0_f:.5f}")

# --- Mapa de calor da densidade, restrito a regiao D ---
Xg_line = np.linspace(a_m, b_m, 250)
ylo_vals = avalia(y_lo_m, (x,), (Xg_line,))
yhi_vals = avalia(y_hi_m, (x,), (Xg_line,))
ymin_plot = float(np.min(ylo_vals))
ymax_plot = float(np.max(yhi_vals))

Xg, Yg = np.meshgrid(np.linspace(a_m, b_m, 200), np.linspace(ymin_plot, ymax_plot, 200))
Rho = avalia(rho_sym, (x, y), (Xg, Yg))

ylo_grid = avalia(y_lo_m, (x,), (Xg[0],))
yhi_grid = avalia(y_hi_m, (x,), (Xg[0],))
dentro = (Yg >= ylo_grid) & (Yg <= yhi_grid)
Rho_masked = np.where(dentro, Rho, np.nan)

fig, ax = plt.subplots(figsize=(8, 6.5), dpi=100, facecolor="w")
cf = ax.contourf(Xg, Yg, Rho_masked, levels=25, cmap="inferno")
fig.colorbar(cf, ax=ax, label="rho(x,y)")
ax.plot(Xg_line, ylo_vals, color="white", linewidth=2)
ax.plot(Xg_line, yhi_vals, color="white", linewidth=2)
ax.set_title(f"Densidade rho(x,y) = {rho_str} sobre a regiao D", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 4. Integrais de Linha e Comprimento de Curvas

Uma curva $C$ parametrizada por $\mathbf r(t)=(x(t),y(t),z(t))$, $t\in[a,b]$, tem vetor velocidade $\mathbf r'(t)$ e **comprimento de arco**:

$$L=\int_a^b |\mathbf r'(t)|\,dt=\int_a^b \sqrt{x'(t)^2+y'(t)^2+z'(t)^2}\,dt$$

A **integral de linha** de um campo escalar $f(x,y,z)$ ao longo de $C$ (em relação ao comprimento de arco) é:

$$\int_C f\,ds=\int_a^b f(x(t),y(t),z(t))\,|\mathbf r'(t)|\,dt$$

In [ ]:
# === Parametros ===
x_t_str, y_t_str, z_t_str = "cos(t)", "sin(t)", "t/(2*pi)"  # r(t) = (x(t), y(t), z(t)) -- helice
t0, t1 = 0.0, float(4 * np.pi)
campo_escalar_str = "1 + z"  # f(x,y,z) para a integral de linha int_C f ds
n_tangentes = 24  # quantidade de vetores tangentes exibidos ao longo da curva
# ==================

x_t, y_t, z_t = parse(x_t_str), parse(y_t_str), parse(z_t_str)
r_sym = sp.Matrix([x_t, y_t, z_t])
rp_sym = r_sym.diff(t)
speed_sym = sp.sqrt(sum(c**2 for c in rp_sym))

print("r(t) =")
display(r_sym.T)
print("r'(t) =")
display(rp_sym.T)
print("|r'(t)| =")
display(sp.simplify(speed_sym))

L = integral_numerica(speed_sym, t, t0, t1)
print(f"\nComprimento de arco:  L = integral de {t0:.3f} a {t1:.3f} de |r'(t)| dt = {L:.5f}")

f_esc_sym = parse(campo_escalar_str)
integrando = f_esc_sym.subs({x: x_t, y: y_t, z: z_t}) * speed_sym
int_linha = integral_numerica(integrando, t, t0, t1)
print(f"Integral de linha:  int_C f ds = {int_linha:.5f}   (f(x,y,z) = {campo_escalar_str})")

ts = np.linspace(t0, t1, 400)
Xc = avalia(x_t, (t,), (ts,))
Yc = avalia(y_t, (t,), (ts,))
Zc = avalia(z_t, (t,), (ts,))

ts_q = np.linspace(t0, t1, n_tangentes)
Xq = avalia(x_t, (t,), (ts_q,))
Yq = avalia(y_t, (t,), (ts_q,))
Zq = avalia(z_t, (t,), (ts_q,))
Uq = avalia(rp_sym[0], (t,), (ts_q,))
Vq = avalia(rp_sym[1], (t,), (ts_q,))
Wq = avalia(rp_sym[2], (t,), (ts_q,))

fig = plt.figure(figsize=(9, 7), dpi=100, facecolor="w")
ax = fig.add_subplot(111, projection="3d")
ax.plot(Xc, Yc, Zc, color="steelblue", linewidth=2.2, label="curva C")
ax.quiver(Xq, Yq, Zq, Uq, Vq, Wq, length=0.25, normalize=True, color="crimson", label="r'(t)")
ax.scatter([Xc[0]], [Yc[0]], [Zc[0]], color="green", s=80, label="inicio (t0)")
ax.scatter([Xc[-1]], [Yc[-1]], [Zc[-1]], color="black", s=80, label="fim (t1)")
ax.set_title(f"Curva r(t) = ({x_t_str}, {y_t_str}, {z_t_str})  —  L = {L:.4f}", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

## 5. Teorema de Green

Para uma curva $C$ fechada, simples, percorrida no sentido anti-horário, delimitando a região $D$, e um campo $\mathbf F=(P,Q)$ com derivadas parciais contínuas:

$$\oint_C P\,dx+Q\,dy=\iint_D\left(\frac{\partial Q}{\partial x}-\frac{\partial P}{\partial y}\right)dA$$

O lado esquerdo é calculado diretamente pela parametrização de $C$; o lado direito, pela integral dupla do rotacional escalar sobre $D$ (aqui, um disco de raio $R$, calculada em coordenadas polares). Os dois valores devem coincidir.

In [ ]:
# === Parametros ===
P_str, Q_str = "-y", "x**2"  # campo F = (P, Q)
R_green = 1.5  # raio do disco D, cuja fronteira C e percorrida no sentido anti-horario
n_amostras = 16  # pontos de amostragem do campo vetorial na visualizacao
# ==================

P_sym, Q_sym = parse(P_str), parse(Q_str)
rot_sym = sp.diff(Q_sym, x) - sp.diff(P_sym, y)

print("Campo F = (P, Q):")
display(sp.Eq(sp.Symbol("P"), P_sym))
display(sp.Eq(sp.Symbol("Q"), Q_sym))
print("\nRotacional escalar  dQ/dx - dP/dy:")
display(sp.Eq(sp.Symbol("rot"), rot_sym))

# Lado esquerdo: integral de linha direta sobre C: (x,y) = (R cos t, R sin t)
xt_g, yt_g = R_green * sp.cos(t), R_green * sp.sin(t)
dxt_g, dyt_g = sp.diff(xt_g, t), sp.diff(yt_g, t)
integrando_linha = (
    P_sym.subs({x: xt_g, y: yt_g}) * dxt_g + Q_sym.subs({x: xt_g, y: yt_g}) * dyt_g
)
linha_green = integral_numerica(sp.simplify(integrando_linha), t, 0, 2 * sp.pi)
print(f"\nLado esquerdo   oint_C P dx + Q dy          = {linha_green:.6f}")

# Lado direito: integral dupla do rotacional sobre o disco D, em coordenadas polares
r_pol, th_pol = sp.symbols("r_pol theta_pol", nonnegative=True, real=True)
rot_polar = rot_sym.subs({x: r_pol * sp.cos(th_pol), y: r_pol * sp.sin(th_pol)}) * r_pol
dupla_green = integral_dupla(rot_polar, r_pol, 0, R_green, th_pol, 0, 2 * sp.pi)
print(f"Lado direito    iint_D (dQ/dx - dP/dy) dA    = {dupla_green:.6f}")
if abs(linha_green - dupla_green) < 1e-6:
    print("Teorema de Green confirmado: os dois lados coincidem!")
else:
    print("Atencao: os valores nao coincidem — revise o campo ou a regiao.")

# --- Visualizacao: campo vetorial + curva C + regiao D ---
lim = R_green * 1.4
Xq, Yq = np.meshgrid(np.linspace(-lim, lim, n_amostras), np.linspace(-lim, lim, n_amostras))
Uq = avalia(P_sym, (x, y), (Xq, Yq))
Vq = avalia(Q_sym, (x, y), (Xq, Yq))

thetas = np.linspace(0, 2 * np.pi, 200)
Xc, Yc = R_green * np.cos(thetas), R_green * np.sin(thetas)

fig, ax = plt.subplots(figsize=(7.5, 7), dpi=100, facecolor="w")
ax.fill(Xc, Yc, color="khaki", alpha=0.35, label="regiao D")
ax.quiver(Xq, Yq, Uq, Vq, color="steelblue", alpha=0.8, width=0.0035)
ax.plot(Xc, Yc, color="crimson", linewidth=2.5, label="curva C (anti-horario)")
idx_seta = len(thetas) // 4
ax.annotate(
    "", xy=(Xc[idx_seta + 2], Yc[idx_seta + 2]), xytext=(Xc[idx_seta], Yc[idx_seta]),
    arrowprops=dict(arrowstyle="->", color="crimson", lw=2.5),
)
ax.set_title(f"F = ({P_str}, {Q_str})  sobre D: x^2+y^2 <= {R_green}^2", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Trabalho de um Campo de Força — Conservativo vs. Não Conservativo

Se $\mathbf F=\nabla\phi$ é **conservativo** (possui função potencial $\phi$), o trabalho realizado ao longo de qualquer curva $C$ de $A$ até $B$ depende apenas dos extremos:

$$W=\int_C \mathbf F\cdot d\mathbf r=\phi(B)-\phi(A)$$

ou seja, o trabalho **independe do caminho**. Já para um campo **não conservativo** ($\partial P/\partial y\ne \partial Q/\partial x$), caminhos diferentes entre os mesmos dois pontos produzem trabalhos diferentes. Comparamos abaixo dois caminhos entre os mesmos pontos, para um campo conservativo e para um campo não conservativo.

In [ ]:
# === Parametros ===
phi_str = "x**2*y + y**3/3"  # funcao potencial phi(x,y); F_conservativo = grad(phi)
P_nc_str, Q_nc_str = "-y", "x"  # campo NAO conservativo F_nc = (P, Q)  (dP/dy != dQ/dx)
A_pt = (0.0, 0.0)  # ponto inicial
B_pt = (1.0, 1.0)  # ponto final
# ==================

phi_sym = parse(phi_str)
Fx_sym, Fy_sym = sp.diff(phi_sym, x), sp.diff(phi_sym, y)
print("Campo conservativo F = grad(phi), com phi(x,y) =")
display(phi_sym)
print("F = (dphi/dx, dphi/dy) =")
display(sp.Matrix([Fx_sym, Fy_sym]).T)

Pnc_sym, Qnc_sym = parse(P_nc_str), parse(Q_nc_str)
checagem = sp.simplify(sp.diff(Pnc_sym, y) - sp.diff(Qnc_sym, x))
print(f"\nCampo nao conservativo F_nc = ({P_nc_str}, {Q_nc_str})")
print(f"dP/dy - dQ/dx = {checagem}  ({'conservativo!' if checagem == 0 else 'NAO conservativo'})")

s = sp.Symbol("s", real=True)


def trabalho(Px_sym, Py_sym, caminho_x, caminho_y, s0, s1):
    '''Calcula o trabalho W = int_C (Px dx + Py dy) ao longo do caminho (x(s), y(s)).'''
    dx_s = sp.diff(caminho_x, s)
    dy_s = sp.diff(caminho_y, s)
    integrando = (
        Px_sym.subs({x: caminho_x, y: caminho_y}) * dx_s
        + Py_sym.subs({x: caminho_x, y: caminho_y}) * dy_s
    )
    return integral_numerica(sp.simplify(integrando), s, s0, s1)


Ax, Ay = A_pt
Bx, By = B_pt

# Caminho 1: reta de A a B
reta_x = Ax + s * (Bx - Ax)
reta_y = Ay + s * (By - Ay)

# Caminho 2: mesmo ponto inicial e final, trajetoria curva (desvio quadratico da reta)
curva_x = Ax + s * (Bx - Ax)
curva_y = Ay + (By - Ay) * s**2 + 0.6 * s * (1 - s)

W_cons_reta = trabalho(Fx_sym, Fy_sym, reta_x, reta_y, 0, 1)
W_cons_curva = trabalho(Fx_sym, Fy_sym, curva_x, curva_y, 0, 1)
W_nc_reta = trabalho(Pnc_sym, Qnc_sym, reta_x, reta_y, 0, 1)
W_nc_curva = trabalho(Pnc_sym, Qnc_sym, curva_x, curva_y, 0, 1)

delta_phi = float(phi_sym.subs({x: Bx, y: By}) - phi_sym.subs({x: Ax, y: Ay}))

print(f"\nphi(B) - phi(A) = {delta_phi:.5f}")
print(f"Conservativo  — caminho reto : W = {W_cons_reta:.5f}")
print(f"Conservativo  — caminho curvo: W = {W_cons_curva:.5f}   (igual ao de cima, como esperado)")
print(f"Nao conserv.  — caminho reto : W = {W_nc_reta:.5f}")
print(f"Nao conserv.  — caminho curvo: W = {W_nc_curva:.5f}   (diferente do de cima!)")

# --- Visualizacao: os dois caminhos + comparacao do trabalho em barras ---
ss = np.linspace(0, 1, 200)
reta_x_num, reta_y_num = avalia(reta_x, (s,), (ss,)), avalia(reta_y, (s,), (ss,))
curva_x_num, curva_y_num = avalia(curva_x, (s,), (ss,)), avalia(curva_y, (s,), (ss,))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=100, facecolor="w")

ax = axes[0]
ax.plot(reta_x_num, reta_y_num, color="steelblue", linewidth=2.5, label="caminho 1 (reta)")
ax.plot(curva_x_num, curva_y_num, color="darkorange", linewidth=2.5, label="caminho 2 (curvo)")
ax.plot(*A_pt, "go", markersize=11, label="A (inicio)")
ax.plot(*B_pt, "ks", markersize=11, label="B (fim)")
ax.set_title("Dois caminhos de A ate B", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="best", fontsize=9)

ax = axes[1]
barras = ["Conserv.\nreta", "Conserv.\ncurvo", "Nao cons.\nreta", "Nao cons.\ncurvo"]
valores = [W_cons_reta, W_cons_curva, W_nc_reta, W_nc_curva]
cores = ["steelblue", "steelblue", "darkorange", "darkorange"]
barras_plot = ax.bar(barras, valores, color=cores, edgecolor="k", alpha=0.85)
for b, val in zip(barras_plot, valores):
    ax.annotate(
        f"{val:.3f}", (b.get_x() + b.get_width() / 2, val), ha="center",
        va="bottom" if val >= 0 else "top", fontsize=9, fontweight="bold",
    )
ax.axhline(0, color="k", linewidth=0.8)
ax.set_title("Trabalho W por caminho e por campo", fontsize=11, fontweight="bold")
ax.set_ylabel("W")

plt.suptitle("Independencia do caminho: conservativo vs. nao conservativo", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Representação Paramétrica de Superfícies

Uma superfície $S$ pode ser descrita por $\mathbf r(u,v)=(x(u,v),y(u,v),z(u,v))$, com $(u,v)\in D\subset\mathbb R^2$. As curvas obtidas fixando $u=u_0$ (ou $v=v_0$) formam a **malha de curvas coordenadas** sobre a superfície — análogas a paralelos e meridianos num globo.

Exemplos clássicos:

| Superfície | $\mathbf r(u,v)$ |
|---|---|
| Plano | $(u,\ v,\ 0)$ |
| Esfera de raio $R$ | $(R\sin u\cos v,\ R\sin u\sin v,\ R\cos u)$ |
| Cone de altura $h$ e raio $r$ | $\left(\dfrac{ru}{h}\cos v,\ \dfrac{ru}{h}\sin v,\ u\right)$ |
| Toro ($R$ = raio maior, $r$ = raio do tubo) | $((R+r\cos u)\cos v,\ (R+r\cos u)\sin v,\ r\sin u)$ |

In [ ]:
# === Parametros ===
superficie = "esfera"  # opcoes: "plano", "esfera", "cone", "toro"
R_esf = 2.0
h_cone, r_cone = 3.0, 1.5
R_tor, r_tor = 2.0, 0.6
n_malha = 45  # resolucao da malha da superficie
n_curvas = 10  # quantidade de curvas u=const e v=const destacadas (grafico interativo)
# ==================


def parametrizacao(tipo):
    '''Retorna (r_sym, (u_min,u_max), (v_min,v_max)) para a superficie escolhida.'''
    if tipo == "plano":
        dom_u, dom_v = (-2.0, 2.0), (-2.0, 2.0)
        r_s = sp.Matrix([u, v, sp.Integer(0)])
    elif tipo == "esfera":
        dom_u, dom_v = (0.0, float(np.pi)), (0.0, float(2 * np.pi))
        r_s = sp.Matrix(
            [R_esf * sp.sin(u) * sp.cos(v), R_esf * sp.sin(u) * sp.sin(v), R_esf * sp.cos(u)]
        )
    elif tipo == "cone":
        dom_u, dom_v = (0.0, h_cone), (0.0, float(2 * np.pi))
        r_s = sp.Matrix(
            [(r_cone * u / h_cone) * sp.cos(v), (r_cone * u / h_cone) * sp.sin(v), u]
        )
    elif tipo == "toro":
        dom_u, dom_v = (0.0, float(2 * np.pi)), (0.0, float(2 * np.pi))
        r_s = sp.Matrix(
            [
                (R_tor + r_tor * sp.cos(u)) * sp.cos(v),
                (R_tor + r_tor * sp.cos(u)) * sp.sin(v),
                r_tor * sp.sin(u),
            ]
        )
    else:
        raise ValueError(f"superficie desconhecida: {tipo}")
    return r_s, dom_u, dom_v


r_sym_sup, (u_min, u_max), (v_min, v_max) = parametrizacao(superficie)
print(f"Parametrizacao escolhida: {superficie}")
print("r(u,v) =")
display(r_sym_sup.T)

Ug, Vg = np.meshgrid(np.linspace(u_min, u_max, n_malha), np.linspace(v_min, v_max, n_malha))
Xp = avalia(r_sym_sup[0], (u, v), (Ug, Vg))
Yp = avalia(r_sym_sup[1], (u, v), (Ug, Vg))
Zp = avalia(r_sym_sup[2], (u, v), (Ug, Vg))

fig = plt.figure(figsize=(9, 7), dpi=100, facecolor="w")
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xp, Yp, Zp, cmap="viridis", alpha=0.75, rstride=2, cstride=2, linewidth=0)
passo = max(1, n_malha // n_curvas)
ax.plot_wireframe(Xp, Yp, Zp, rstride=passo, cstride=passo, color="k", linewidth=0.5, alpha=0.6)
ax.set_title(f"Superficie parametrica: {superficie}", fontsize=12, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.tight_layout()
plt.show()

if PLOTLY:
    fig_pl = go.Figure(
        data=[go.Surface(x=Xp, y=Yp, z=Zp, colorscale="Viridis", opacity=0.92, showscale=False)]
    )
    for u0 in np.linspace(u_min, u_max, n_curvas):
        Vl = np.linspace(v_min, v_max, 100)
        Ul = np.full_like(Vl, u0)
        Xl = avalia(r_sym_sup[0], (u, v), (Ul, Vl))
        Yl = avalia(r_sym_sup[1], (u, v), (Ul, Vl))
        Zl = avalia(r_sym_sup[2], (u, v), (Ul, Vl))
        fig_pl.add_trace(
            go.Scatter3d(x=Xl, y=Yl, z=Zl, mode="lines", line=dict(color="black", width=2), showlegend=False)
        )
    for v0 in np.linspace(v_min, v_max, n_curvas):
        Ul = np.linspace(u_min, u_max, 100)
        Vl = np.full_like(Ul, v0)
        Xl = avalia(r_sym_sup[0], (u, v), (Ul, Vl))
        Yl = avalia(r_sym_sup[1], (u, v), (Ul, Vl))
        Zl = avalia(r_sym_sup[2], (u, v), (Ul, Vl))
        fig_pl.add_trace(
            go.Scatter3d(x=Xl, y=Yl, z=Zl, mode="lines", line=dict(color="white", width=2), showlegend=False)
        )
    fig_pl.update_layout(
        title=f"Superficie interativa: {superficie}  (malha de curvas u=const, v=const)",
        scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
        margin=dict(l=0, r=0, b=0, t=45),
        height=600,
    )
    fig_pl.show()
else:
    print("Instale plotly para visualizacao 3-D interativa: pip install plotly")

## 8. Integrais de Superfície

Dada $\mathbf r(u,v)$, o **elemento de área** da superfície é a norma do produto vetorial fundamental:

$$dS=|\mathbf r_u\times\mathbf r_v|\,du\,dv,\qquad \mathbf r_u=\frac{\partial \mathbf r}{\partial u},\ \ \mathbf r_v=\frac{\partial \mathbf r}{\partial v}$$

A **área da superfície** e a **integral de uma função escalar $f$ sobre $S$** são:

$$A=\iint_D |\mathbf r_u\times\mathbf r_v|\,dA,\qquad \iint_S f\,dS=\iint_D f(\mathbf r(u,v))\,|\mathbf r_u\times\mathbf r_v|\,dA$$

O vetor $\mathbf r_u\times\mathbf r_v$ também fornece a **direção normal** à superfície em cada ponto — usa a parametrização escolhida na Seção 7 (troque `superficie` lá e reexecute as duas células).

In [ ]:
# === Parametros ===
# reaproveita a parametrizacao escolhida na Secao 7 (variavel r_sym_sup e os limites
# u_min, u_max, v_min, v_max) -- troque 'superficie' na Secao 7 e reexecute as duas celulas
funcao_escalar_str = "1"  # f(x,y,z) para a integral de superficie iint_S f dS ("1" = area)
n_normais = 10  # quantidade de vetores normais exibidos
# ==================

ru_sym = r_sym_sup.diff(u)
rv_sym = r_sym_sup.diff(v)
normal_sym = ru_sym.cross(rv_sym)
jac_sym = sp.simplify(sp.sqrt((normal_sym.T * normal_sym)[0]))

print(f"Superficie: {superficie}")
print("r_u = dr/du =")
display(ru_sym.T)
print("r_v = dr/dv =")
display(rv_sym.T)
print("r_u x r_v =")
display(normal_sym.T)
print("|r_u x r_v|  (elemento de area dS) =")
display(jac_sym)

area_val = integral_dupla(jac_sym, v, v_min, v_max, u, u_min, u_max)
print(f"\nArea da superficie:  A = {area_val:.5f}")

f_sup_sym = parse(funcao_escalar_str)
integrando_sup = f_sup_sym.subs({x: r_sym_sup[0], y: r_sym_sup[1], z: r_sym_sup[2]}) * jac_sym
int_sup_val = integral_dupla(integrando_sup, v, v_min, v_max, u, u_min, u_max)
print(f"Integral de superficie:  iint_S f dS = {int_sup_val:.5f}   (f(x,y,z) = {funcao_escalar_str})")

# --- Visualizacao: superficie + vetores normais ---
Ug2, Vg2 = np.meshgrid(np.linspace(u_min, u_max, n_malha), np.linspace(v_min, v_max, n_malha))
Xp2 = avalia(r_sym_sup[0], (u, v), (Ug2, Vg2))
Yp2 = avalia(r_sym_sup[1], (u, v), (Ug2, Vg2))
Zp2 = avalia(r_sym_sup[2], (u, v), (Ug2, Vg2))

Uq2, Vq2 = np.meshgrid(np.linspace(u_min, u_max, n_normais), np.linspace(v_min, v_max, n_normais))
Xq2 = avalia(r_sym_sup[0], (u, v), (Uq2, Vq2))
Yq2 = avalia(r_sym_sup[1], (u, v), (Uq2, Vq2))
Zq2 = avalia(r_sym_sup[2], (u, v), (Uq2, Vq2))
Nx2 = avalia(normal_sym[0], (u, v), (Uq2, Vq2))
Ny2 = avalia(normal_sym[1], (u, v), (Uq2, Vq2))
Nz2 = avalia(normal_sym[2], (u, v), (Uq2, Vq2))

fig = plt.figure(figsize=(9, 7), dpi=100, facecolor="w")
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(Xp2, Yp2, Zp2, cmap="viridis", alpha=0.65, rstride=2, cstride=2, linewidth=0)
ax.quiver(Xq2, Yq2, Zq2, Nx2, Ny2, Nz2, length=0.35, normalize=True, color="crimson")
ax.set_title(f"Vetor normal r_u x r_v sobre a superficie ({superficie})", fontsize=11, fontweight="bold")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.tight_layout()
plt.show()